# Shakespeare task

The task is to implement a simple RNN using Pytorch and Python,

getting the model to learn from the Shakespeare complete works

data set so it can then generate text in the same style.

Below I am importing the required libraries.

In [1]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import requests

# URL of the Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Fetching the text data from the URL
response = requests.get(url)
text = response.text

# Printing the length of the fetched text
print("Length of text: ", len(text))

Length of text:  1115394


We now need to create a dictionary of each inidivual letter in the text.

Above we can see that there are 1.1mil letters, spaces and symbols in the text.

set(text) takes each unique letter (in upper and lower case), space and symbol

and chars is a sorted list of these items.

char_indices makes a dictionary for each of these symbols/letters/spaces where

the index is a number.

indices_char is the inverse for when we need to map a number to a letter or

symbol.

In [2]:
# Creating character mappings
chars = sorted(list(set(text)))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

We now generate a sequence of characters. maxlen defines the length and

of the window and step defines how many spaces the window moves with each step.

This creates overlapping sequences of characters from the text.

If the length of the window is too small it doesn't learn the sentence

structure of the text very well, if it's too long it suffers from a

vanishing gradient problem where early characters stop influencing the weights.

Sentence is the 40 characters from within the maxlen window and next_chars

stores the character each time that directly follows the end of the window.

For the first run sentences would be the first 40 characters and next_chars

would be the 41st.

We result in 371,785 samples of sentences and the subsequent character

next_chars for the model to train on.

In [3]:
# Creating sequences
maxlen = 40
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])
print("Number of sequences:", len(sentences))


Number of sequences: 371785


We now vectorize the data for the model to train on.

X is a vector of vectors as it is one hot encoding each character in the

window. X's shape is in this instance is [371,785, 40, 65] as there are 55,

rows of vectors and each vector has a shape of 40 by 65. 40 being the number of

characters in each window and 65 being the total number of unique characters

in the text. Each of the 40 rows will have zeroes in each column apart from

that has a 1 in it. This will be unique for this character.

y is only one character each time so it's shape will be [371,785, 65].

In [4]:
# Vectorizing the data
X = torch.zeros((len(sentences), maxlen, len(chars)), dtype=torch.float32)
y = torch.zeros((len(sentences), len(chars)), dtype=torch.float32)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

In [5]:
print(X.shape)

torch.Size([371785, 40, 65])


Now that the data is in a format the model can learn from we will create the

model itself. Below is an RNN Model where the input and output size are

define by the number of unique characters in the body of text, in our case 65.

The below model only has 1 hidden layer and we have defined it will have 128

nodes.

In the forward pass the RNN processes each character in the window one at a

time, so the "out = self.fc(out[:, -1, :])" section is us only taking the final

RNN output as by then it's learned from all the characters in the sequence

has the most complete understanding of the sequence.


In [6]:
# Building the model
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(RNNModel, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)


    def forward(self, x):
        out, (h_n, c_n) = self.rnn(x)  
        out = self.fc(out[:, -1, :])
        return out

In [7]:
input_size = len(chars)
hidden_size = 128
output_size = len(chars)

model = RNNModel(input_size, hidden_size, output_size, num_layers=2)

Before the learning phase we choose the loss function to evaluate how well

the model's predictions match  the actual data.

CrossEntropyLoss takes the 65 scores the model creates and applies softmax

so their scores sum to 1. It then compares the result to the actual answer

to calculate the loss. This impacts the re-weightings for backpropogation.

The Adam optimizer has been selected to adjust the models parameters during

training. This optimizer is an advanced optimization algorithm that adapts

the learning rate for each parameter by updating running averages of both

the gradients and the squares.

We've selected a learning rate of 0.01.

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

The learning phase starts by training a model over 40 epochs using mini

batches of 128 samples each. In each batch the model performs a forward pass

to predict the outputs and compute the loss. It then performs a backward

pass to calculate the gradients and update the model parameters using the

optimizer.

In [9]:
# Training the model
num_epochs = 40
batch_size = 128
for epoch in range(num_epochs):
    for i in range(0, len(X), batch_size):
        X_batch = X[i:i+batch_size]
        y_batch = y[i:i+batch_size]

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/40], Loss: 2.0066
Epoch [2/40], Loss: 1.8368
Epoch [3/40], Loss: 1.7352
Epoch [4/40], Loss: 1.6302
Epoch [5/40], Loss: 1.5438
Epoch [6/40], Loss: 1.4699
Epoch [7/40], Loss: 1.4050
Epoch [8/40], Loss: 1.3536
Epoch [9/40], Loss: 1.3213
Epoch [10/40], Loss: 1.2957
Epoch [11/40], Loss: 1.2497
Epoch [12/40], Loss: 1.2061
Epoch [13/40], Loss: 1.1820
Epoch [14/40], Loss: 1.1813
Epoch [15/40], Loss: 1.1699
Epoch [16/40], Loss: 1.1346
Epoch [17/40], Loss: 1.1175
Epoch [18/40], Loss: 1.0950
Epoch [19/40], Loss: 1.0782
Epoch [20/40], Loss: 1.0612
Epoch [21/40], Loss: 1.0321
Epoch [22/40], Loss: 1.0303
Epoch [23/40], Loss: 0.9944
Epoch [24/40], Loss: 0.9623
Epoch [25/40], Loss: 0.9373
Epoch [26/40], Loss: 0.8992
Epoch [27/40], Loss: 0.9088
Epoch [28/40], Loss: 0.9026
Epoch [29/40], Loss: 0.8907
Epoch [30/40], Loss: 0.8412
Epoch [31/40], Loss: 0.8436
Epoch [32/40], Loss: 0.8204
Epoch [33/40], Loss: 0.8274
Epoch [34/40], Loss: 0.8265
Epoch [35/40], Loss: 0.8209
Epoch [36/40], Loss: 0.8877
E

The loss value improves across the 40 epochs and so it does learn

through the epoc's to improve its performance.

We are now going to create a function that can use the model to generate 100

characters of text in the style of Shakespeare. This works better when we

provide the model some seed text to start it off, I've chosen a quote from

Bill and Ted's excellent adventure to test it.

The function is initiated by importing the variables, and I've set the word

count to be 100 so the loop stops generating when it hits the required amount.

The seed text is less than 40 characters, so since the model is configured

to look at the previous 40 characters and predict the next one I've padded

out the seed text to make it 40 characters long.

The current seq starts as being the seed text.

The function vectorizes the previous 40 characters and uses the model to

output a prediction. To create the x vector we're doing this in batches of 1

so the first number is 1. I've added a section at the end to catch for if a

new character is added that's not included in the char_to_idx. If that does

happen it will add a new pair to the char_indices dictionary.

Next we run the vector through the model to get an output, which will tell

us which character has the highest probability of being next in the sequence.

The argmax sampling chooses the character with the highest probability from

the output, we then use the idx_to_char to map index to the character.

I then add the character to the end of the text and move the window along

one step.

In [10]:
def generate_text(model, seed_text, char_to_idx, idx_to_char, max_len,
                  word_count=100):
    model.eval()
    seed_text = seed_text.rjust(max_len)
    generated = seed_text
    current_seq = seed_text

    with (torch.no_grad()):
        while len(generated.split()) < word_count:
            #1. Vectorize the last 40 characters in the window
            x = torch.zeros((1, max_len, len(char_to_idx)), dtype=torch.float32)
            for i, char in enumerate(current_seq[-max_len:]):
                if char in char_to_idx:
                    x[0, i, char_to_idx[char]] = 1

            #2. Get model prediction
            output = model(x)  # shape: (1, num_chars)
            output = torch.softmax(output, dim=1)

            # 3. Sample from the distribution
            predicted_idx = torch.multinomial(output, num_samples=1).item()
            predicted_char = idx_to_char[predicted_idx]

            # 4. Append predicted char and slide the window
            generated += predicted_char
            current_seq = current_seq[1:] + predicted_char

    return generated

This is running the function with the input variables and printing the output.

In [12]:
output = generate_text(
    model=model,
    seed_text="All we are is dust in the wind, dude.",
    char_to_idx= char_indices,
    idx_to_char=indices_char,
    max_len=40,
    word_count=100
)
print(output)

   All we are is dust in the wind, dude.

GONZOOF:
Nobbly mean, and I fare-park'st thou tinle,
And where is worst to wonder thine eye will it:
Prominel that I bestrived both of my pass;
For is did, planing stee and but a vurity
Ashounding beseech me contrus you apposar.
To these what is the least.

SEBASTIAN:
Watgures have been days you'll king.

Mestenkmend:
They rave the envalment: woilo's, I prictsa
Hath make misturp: you be but tear.

HORTENSIO:
If-coural at it be.

PETRUCHIO:
What said thy frown?

MIRANDA:
Forwell, good Biendel, promated?

PETRUCHIO:
I am contrnain, when the h


The model produced a passage of text that has Shakespearian overtones.

The text is structured as a conversation between various parties and the words

by and large make sense, so I would consider this a success, even if the

story isn't brilliantly compelling.